# RuleShift Benchmark v1 — Kaggle Benchmark

## Runtime bootstrap

In [ ]:
from collections import Counter
from importlib import import_module
from importlib.metadata import PackageNotFoundError, version
import sys
import time
from pathlib import Path

DEBUG_MODE = False
DEBUG_LIMIT = None
DEBUG_FAIL_ON_UNSCOREABLE = True

import kaggle_benchmarks as kbench


def _find_repo_root() -> Path:
    candidates: list[Path] = []
    seen: set[Path] = set()

    for origin in (Path.cwd().resolve(),):
        for candidate in (origin, *origin.parents):
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for search_root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not search_root.exists():
            continue
        for manifest_path in search_root.rglob("frozen_artifacts_manifest.json"):
            candidate = manifest_path.parents[2]
            if candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    for candidate in candidates:
        if (candidate / "src").is_dir() and (
            candidate / "packaging" / "kaggle" / "frozen_artifacts_manifest.json"
        ).is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate repo root. Expected src/ and "
        "packaging/kaggle/frozen_artifacts_manifest.json."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))


## Frozen leaderboard split

In [ ]:
from core.kaggle import (
    build_kaggle_payload,
    load_leaderboard_dataframe,
    normalize_count_result_df,
    run_binary_task,
    validate_kaggle_payload,
)
from core.kaggle.runner import (
    BinaryResponse,
    KaggleExecutionError,
    normalize_binary_response,
    score_episode,
)


def _resolve_installed_version(*distribution_names: str) -> str:
    for distribution_name in distribution_names:
        try:
            return version(distribution_name)
        except PackageNotFoundError:
            continue
    return "not installed as a package"


def _preview_response(response: object, *, limit: int = 120) -> str:
    preview = "None" if response is None else repr(response)
    preview = " ".join(preview.split())
    if len(preview) > limit:
        return preview[: limit - 3] + "..."
    return preview


def _response_type_name(response: object) -> str:
    return type(response).__name__


class DebugLLM:
    def __init__(self, wrapped_llm: object) -> None:
        self._wrapped_llm = wrapped_llm
        self.calls: list[dict[str, object]] = []

    def __getattr__(self, name: str) -> object:
        return getattr(self._wrapped_llm, name)

    def prompt(self, text: str, *, schema: object = None) -> object:
        call_index = len(self.calls) + 1
        started_at = time.perf_counter()
        prompt_size = len(text)
        try:
            response = self._wrapped_llm.prompt(text, schema=schema)
        except Exception as exc:
            latency_ms = (time.perf_counter() - started_at) * 1000.0
            entry = {
                "call_index": call_index,
                "latency_ms": latency_ms,
                "prompt_size": prompt_size,
                "response_type": f"EXC:{type(exc).__name__}",
                "response_preview": _preview_response(exc),
            }
            self.calls.append(entry)
            print(
                f"[DEBUG] llm.prompt call #{call_index} raised {type(exc).__name__} "
                f"after {latency_ms:.1f} ms | prompt_size={prompt_size}"
            )
            raise

        latency_ms = (time.perf_counter() - started_at) * 1000.0
        response_type = _response_type_name(response)
        response_preview = _preview_response(response)
        entry = {
            "call_index": call_index,
            "latency_ms": latency_ms,
            "prompt_size": prompt_size,
            "response_type": response_type,
            "response_preview": response_preview,
        }
        self.calls.append(entry)
        print(
            f"[DEBUG] llm.prompt call #{call_index}: {latency_ms:.1f} ms | "
            f"prompt_size={prompt_size} | response_type={response_type} | "
            f"preview={response_preview}"
        )
        return response

    def print_summary(self) -> None:
        print("=== Debug LLM Call Summary ===")
        if not self.calls:
            print("total_calls=0")
            print("=== End Debug LLM Call Summary ===")
            return

        latencies_ms = [float(call["latency_ms"]) for call in self.calls]
        response_types = Counter(str(call["response_type"]) for call in self.calls)
        print(f"total_calls={len(self.calls)}")
        print(
            "latency_ms_min_avg_max="
            f"{min(latencies_ms):.1f}/{sum(latencies_ms) / len(latencies_ms):.1f}/{max(latencies_ms):.1f}"
        )
        print(f"response_types={dict(response_types)}")
        print("=== End Debug LLM Call Summary ===")


def run_binary_task_debug(
    *,
    llm: object,
    prompt_binary: str,
    probe_targets: tuple[str, ...],
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    try:
        response = llm.prompt(prompt_binary, schema=BinaryResponse)
    except Exception as exc:
        raise KaggleExecutionError("llm.prompt failed before producing a scoreable response") from exc

    normalized_response = normalize_binary_response(response)
    if normalized_response is None and DEBUG_FAIL_ON_UNSCOREABLE:
        raise ValueError(
            "DEBUG_FAIL_ON_UNSCOREABLE: received an unscoreable model response "
            f"for episode_id={episode_id!r}, split={split!r}, "
            f"response_type={_response_type_name(response)}, prompt_size={len(prompt_binary)}, "
            f"response_preview={_preview_response(response)}"
        )

    return score_episode(normalized_response, probe_targets)


RUNNER_MODULE = import_module("core.kaggle.runner")
RUNNER_MODULE_PATH = getattr(RUNNER_MODULE, "__file__", "<unknown>")
KAGGLE_BENCHMARKS_VERSION = _resolve_installed_version("kaggle_benchmarks", "kaggle-benchmarks")

PRIVATE_DATASET_ROOT, frozen_splits, leaderboard_df = load_leaderboard_dataframe(
    repo_root=REPO_ROOT,
)


## Official task

In [ ]:
@kbench.task(
    name="ruleshift_benchmark_v1_binary_row",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
    store_task=False,
)
def _ruleshift_benchmark_v1_binary_row(
    llm,
    prompt_binary: str,
    probe_targets: tuple,
    split: str,
    episode_id: str | None = None,
) -> tuple[int, int]:
    if DEBUG_MODE:
        return run_binary_task_debug(
            llm=llm,
            prompt_binary=prompt_binary,
            probe_targets=probe_targets,
            split=split,
            episode_id=episode_id,
        )
    del split
    del episode_id
    return run_binary_task(
        llm=llm,
        prompt_binary=prompt_binary,
        probe_targets=probe_targets,
    )


@kbench.task(
    name="ruleshift_benchmark_v1_binary",
    description=(
        "Cognitive flexibility benchmark: infer a hidden rule shift from "
        "sparse contradictory evidence in a sequence of charge interactions, "
        "then predict four post-shift probe outcomes."
    ),
)
def ruleshift_benchmark_v1_binary(llm) -> dict:
    import json

    eval_df = leaderboard_df[["episode_id", "split", "prompt_binary", "probe_targets"]].copy()
    if DEBUG_MODE and DEBUG_LIMIT is not None:
        eval_df = eval_df.head(DEBUG_LIMIT).copy()

    effective_llm = DebugLLM(llm) if DEBUG_MODE else llm
    if DEBUG_MODE:
        print("=== RuleShift Benchmark v1 — Debug Runtime ===")
        print(f"REPO_ROOT={REPO_ROOT}")
        print(f"core.kaggle.runner={RUNNER_MODULE_PATH}")
        print(f"kaggle_benchmarks_version={KAGGLE_BENCHMARKS_VERSION}")
        print(f"evaluation_rows={len(eval_df)}")
        print(f"splits={eval_df['split'].value_counts().to_dict()}")
        print(f"debug_limit={DEBUG_LIMIT}")
        print(f"debug_fail_on_unscoreable={DEBUG_FAIL_ON_UNSCOREABLE}")

    try:
        binary_results = _ruleshift_benchmark_v1_binary_row.evaluate(
            llm=[effective_llm],
            evaluation_data=eval_df,
        )
    finally:
        if DEBUG_MODE:
            effective_llm.print_summary()

    binary_df = normalize_count_result_df(
        binary_results.as_dataframe(),
        dataframe_label="binary_df",
    )
    payload = build_kaggle_payload(binary_df=binary_df)
    validate_kaggle_payload(payload)

    print("=== RuleShift Benchmark v1 — Canonical Result ===")
    print(json.dumps(payload, indent=2))
    print("=== End Canonical Result ===")
    return payload


## Official payload

In [ ]:
payload = ruleshift_benchmark_v1_binary(kbench.llm)


## Task selection

In [ ]:
%choose ruleshift_benchmark_v1_binary